# NB02 – Data Transformation

## European Air Quality and Weather

### Purpose

This notebook transforms the raw API responses collected in NB01 into a clean dataset suitable for analysis.

The notebook extracts relevant variables, converts the nested JSON structure into a flat table, checks data quality, creates additional descriptive variables, and saves the processed dataset for use in NB03.

## First step is to add imports and project folders

In [11]:
import json
from pathlib import Path

import pandas as pd

In [12]:
PROJECT_DIR = Path("..")

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

## Find the newest JSON file

Instead of hard coding the filename, I'll automatically use the newest one.

In [13]:
raw_files = sorted(
    RAW_DATA_DIR.glob("*.json")
)

latest_file = raw_files[-1]

print(latest_file.name)

european_air_quality_weather_20260730_153905.json


## Load the JSON

In [14]:
with open(latest_file, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print(len(raw_data))

10


## Inspect the structure of the data

In [15]:
raw_data[0].keys()

dict_keys(['city_requested', 'country', 'latitude', 'longitude', 'collected_at_utc', 'weather', 'air_pollution'])

## Transforming nested JSON

The API responses contain nested dictionaries for weather and air quality measurements.

These nested values are extracted into a single flat table where each row represents one city. 

## Flatten the data

In [16]:
clean_rows = []

for record in raw_data:

    weather = record["weather"]

    air = record["air_pollution"]["list"][0]

    components = air["components"]

    clean_rows.append({

        "city": record["city_requested"],

        "country": record["country"],

        "collection_time": record["collected_at_utc"],

        "latitude": record["latitude"],

        "longitude": record["longitude"],

        "temperature_c": weather["main"]["temp"],

        "humidity_percent": weather["main"]["humidity"],

        "pressure_hpa": weather["main"]["pressure"],

        "wind_speed_ms": weather["wind"]["speed"],

        "aqi": air["main"]["aqi"],

        "pm2_5": components["pm2_5"],

        "pm10": components["pm10"],

        "no2": components["no2"],

        "o3": components["o3"],

        "co": components["co"]

    })

## Create the Data Frame

In [17]:
df = pd.DataFrame(clean_rows)

df.head()

,city,country,collection_time,latitude,longitude,temperature_c,humidity_percent,pressure_hpa,wind_speed_ms,aqi,pm2_5,pm10,no2,o3,co
0,London,GB,2026-07-30T15:39:05.094096+00:00,51.5074,-0.1278,27.75,45,1014,1.79,2,6.21,14.71,5.14,88.45,96.55
1,Paris,FR,2026-07-30T15:39:05.179592+00:00,48.8566,2.3522,30.24,51,1015,3.01,3,8.87,9.97,0.76,106.30,116.18
2,Berlin,DE,2026-07-30T15:39:05.263652+00:00,52.5200,13.4050,37.23,21,1009,3.58,3,4.79,5.53,3.00,123.64,101.60
3,Madrid,ES,2026-07-30T15:39:05.353611+00:00,40.4168,-3.7038,37.48,17,1015,0.89,3,18.97,77.85,0.20,86.31,91.82
4,Rome,IT,2026-07-30T15:39:05.443955+00:00,41.9028,12.4964,36.16,33,1015,2.68,3,11.66,22.97,0.35,117.59,170.53


## Check the dimensions

In [18]:
print(df.shape)

df.info()

(10, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   city              10 non-null     object 
 1   country           10 non-null     object 
 2   collection_time   10 non-null     object 
 3   latitude          10 non-null     float64
 4   longitude         10 non-null     float64
 5   temperature_c     10 non-null     float64
 6   humidity_percent  10 non-null     int64  
 7   pressure_hpa      10 non-null     int64  
 8   wind_speed_ms     10 non-null     float64
 9   aqi               10 non-null     int64  
 10  pm2_5             10 non-null     float64
 11  pm10              10 non-null     float64
 12  no2               10 non-null     float64
 13  o3                10 non-null     float64
 14  co                10 non-null     float64
dtypes: float64(9), int64(3), object(3)
memory usage: 1.3+ KB


## Check missing values

In [19]:
df.isna().sum()

city                0
country             0
collection_time     0
latitude            0
longitude           0
temperature_c       0
humidity_percent    0
pressure_hpa        0
wind_speed_ms       0
aqi                 0
pm2_5               0
pm10                0
no2                 0
o3                  0
co                  0
dtype: int64

## Duplicate the rows

In [20]:
df.duplicated().sum()

np.int64(0)

## Summary of statistics
This helps check for impossible values before analysis 

In [21]:
df.describe()

,latitude,longitude,temperature_c,humidity_percent,pressure_hpa,wind_speed_ms,aqi,pm2_5,pm10,no2,o3,co
count,10.00000,10.000000,10.000000,10.000000,10.000000,10.00000,10.000000,10.000000,10.000000,10.00000,10.000000,10.000000
mean,49.23813,7.705770,31.177000,42.400000,1014.000000,2.59200,2.800000,9.247000,18.121000,2.85400,105.624000,114.022000
std,4.75267,6.982784,5.521872,20.855055,2.260777,1.47386,0.421637,4.309143,21.639952,2.51103,13.533743,24.098362
min,40.41680,-3.703800,22.230000,17.000000,1009.000000,0.89000,2.000000,3.720000,4.370000,0.20000,86.310000,91.820000
25%,48.37030,2.852075,28.055000,24.000000,1014.000000,1.14500,3.000000,6.562500,8.770000,0.87500,95.557500,97.812500
50%,50.46290,8.700250,30.620000,42.000000,1015.000000,2.68000,3.000000,9.055000,11.380000,2.24000,105.620000,106.975000
75%,52.15255,13.195825,36.377500,50.750000,1015.000000,3.43750,3.000000,10.827500,14.702500,4.60500,115.497500,122.337500
max,55.67610,16.373800,37.480000,83.000000,1016.000000,4.74000,3.000000,18.970000,77.850000,7.01000,123.940000,170.530000


## Create AQI Labels 
The API returns values 1 through 5, this code will make them readable

In [22]:
aqi_labels = {

    1: "Good",

    2: "Fair",

    3: "Moderate",

    4: "Poor",

    5: "Very Poor"

}

df["aqi_category"] = df["aqi"].map(aqi_labels)

## Reorder the chart columns

In [23]:
df = df[

    [

        "city",

        "country",

        "collection_time",

        "latitude",

        "longitude",

        "aqi",

        "aqi_category",

        "pm2_5",

        "pm10",

        "no2",

        "o3",

        "co",

        "temperature_c",

        "humidity_percent",

        "pressure_hpa",

        "wind_speed_ms"

    ]

]

## View the cleaned table

In [26]:
df

,city,country,collection_time,latitude,longitude,aqi,aqi_category,pm2_5,pm10,no2,o3,co,temperature_c,humidity_percent,pressure_hpa,wind_speed_ms
0,London,GB,2026-07-30T15:39:05.094096+00:00,51.5074,-0.1278,2,Fair,6.21,14.71,5.14,88.45,96.55,27.75,45,1014,1.79
1,Paris,FR,2026-07-30T15:39:05.179592+00:00,48.8566,2.3522,3,Moderate,8.87,9.97,0.76,106.30,116.18,30.24,51,1015,3.01
2,Berlin,DE,2026-07-30T15:39:05.263652+00:00,52.5200,13.4050,3,Moderate,4.79,5.53,3.00,123.64,101.60,37.23,21,1009,3.58
3,Madrid,ES,2026-07-30T15:39:05.353611+00:00,40.4168,-3.7038,3,Moderate,18.97,77.85,0.20,86.31,91.82,37.48,17,1015,0.89
4,Rome,IT,2026-07-30T15:39:05.443955+00:00,41.9028,12.4964,3,Moderate,11.66,22.97,0.35,117.59,170.53,36.16,33,1015,2.68
5,Amsterdam,NL,2026-07-30T15:39:05.539197+00:00,52.3676,4.9041,3,Moderate,10.96,14.68,7.01,102.66,124.39,24.26,64,1015,0.89
6,Brussels,BE,2026-07-30T15:39:05.631234+00:00,50.8503,4.3517,2,Fair,9.24,11.28,1.22,93.19,107.92,22.23,83,1016,4.74
7,Vienna,AT,2026-07-30T15:39:05.723319+00:00,48.2082,16.3738,3,Moderate,7.62,8.37,2.82,109.22,106.03,31.00,39,1016,0.93
8,Copenhagen,DK,2026-07-30T15:39:05.812577+00:00,55.6761,12.5683,3,Moderate,10.43,11.48,6.38,123.94,133.38,28.97,50,1011,4.73
9,Prague,CZ,2026-07-30T15:39:05.908949+00:00,50.0755,14.4378,3,Moderate,3.72,4.37,1.66,104.94,91.82,36.45,21,1014,2.68


## Save the processed data

In [27]:
output_file = PROCESSED_DATA_DIR / "air_quality_weather_clean.csv"

df.to_csv(output_file, index=False)

print(output_file.resolve())

/files/assignments/final-project/data/processed/air_quality_weather_clean.csv


## Verify the saved file

In [28]:
check_df = pd.read_csv(output_file)

check_df.head()

,city,country,collection_time,latitude,longitude,aqi,aqi_category,pm2_5,pm10,no2,o3,co,temperature_c,humidity_percent,pressure_hpa,wind_speed_ms
0,London,GB,2026-07-30T15:39:05.094096+00:00,51.5074,-0.1278,2,Fair,6.21,14.71,5.14,88.45,96.55,27.75,45,1014,1.79
1,Paris,FR,2026-07-30T15:39:05.179592+00:00,48.8566,2.3522,3,Moderate,8.87,9.97,0.76,106.30,116.18,30.24,51,1015,3.01
2,Berlin,DE,2026-07-30T15:39:05.263652+00:00,52.5200,13.4050,3,Moderate,4.79,5.53,3.00,123.64,101.60,37.23,21,1009,3.58
3,Madrid,ES,2026-07-30T15:39:05.353611+00:00,40.4168,-3.7038,3,Moderate,18.97,77.85,0.20,86.31,91.82,37.48,17,1015,0.89
4,Rome,IT,2026-07-30T15:39:05.443955+00:00,41.9028,12.4964,3,Moderate,11.66,22.97,0.35,117.59,170.53,36.16,33,1015,2.68


## Data Transformation Summary

The raw API responses were successfully transformed into a tidy dataset.

The transformation process included:

- extracting nested weather measurements
- extracting air-quality measurements
- checking for missing values
- checking for duplicate observations
- creating descriptive AQI categories
- saving the cleaned data as a CSV file

The processed dataset will be used for all visualizations and statistical summaries in NB03.